# Preparing the dataset (ETLCDB - ETL9G)

## 1. Download .zip

In [ ]:
!curl -L -o unpack_etlcdb.zip "http://etlcdb.db.aist.go.jp/download/338/?tmstv=1776445849"

## 2. Unzip

In [ ]:
!unzip ./unpack_etlcdb.zip -d ./etlcdb

## 3. Download ETL9G

In [ ]:
%cd /content/etlcdb/unpack_etlcdb/
!pip install -r requirements.txt

In [ ]:
!curl -L -o ETL9G.zip "http://etlcdb.db.aist.go.jp/download/335/?tmstv=1776566691"

In [ ]:
!unzip ETL9G.zip
!ls -R ETL9G

In [ ]:
%cd /content/etlcdb/unpack_etlcdb/
%rm ETL9G.zip
!python unpack.py ETL9G/ETL9G_01

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import os

# Define paths for the first unpacked volume
output_dir = '/content/etlcdb/unpack_etlcdb/ETL9G/ETL9G_01_unpack'
meta_file = os.path.join(output_dir, 'meta.csv')

if os.path.exists(meta_file):
    # Load metadata
    df_meta = pd.read_csv(meta_file)
    print(f"Total records: {len(df_meta)}")

    # Display first few rows
    display(df_meta.head())

    # Display a sample image (00001.png corresponds to the first row in the CSV)
    sample_img_path = os.path.join(output_dir, '00003.png')
    if os.path.exists(sample_img_path):
        img = Image.open(sample_img_path)
        plt.figure(figsize=(4,4))
        plt.imshow(img, cmap='gray')

        # Use the unicode hex string in title to avoid font issues,
        # while printing the character to standard output where Colab handles it better.
        char_info = df_meta.iloc[3]
        plt.title(f"Unicode: {char_info['unicode']}")
        plt.axis('off')
        plt.show()

        print(f"The character shown above is: {char_info['char']}")
else:
    print("Metadata file not found.")

In [ ]:
if os.path.exists(sample_img_path):
    img = Image.open(sample_img_path)
    width, height = img.size
    print(f"Image size (Width x Height): {width} x {height} pixels")
    print(f"Image mode: {img.mode}")

# Model

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from typing import Optional

class JapaneseCharacterClassifier(nn.Module):
  """CNN model for Japanese character classification"""

  def __init__(self, num_classes: int = 12144):
    super().__init__()
    # Input: 1 x 128 x 127
    self.features = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )

    # After 3 rounds of MaxPool2d(2), size is 128/8 x 127/8 approx 16 x 15
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128 * 16 * 15, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes)
    )

  def forward(self, x):
    # If x is flat, reshape it. Dataset loader provides (Batch, 1, 128, 127)
    if len(x.shape) == 2:
        x = x.view(-1, 1, 128, 127)
    x = self.features(x)
    x = self.classifier(x)
    return x

def load_model(model_path: str, num_classes: int = 12144, device: str = "cpu") -> JapaneseCharacterClassifier:
  """Load a trained model"""
  model = JapaneseCharacterClassifier(num_classes=num_classes)
  try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
  except FileNotFoundError:
    print(f"Warning: Model file not found at {model_path}. Using untrained model.")
  return model

In [ ]:
import torch
import numpy as np
import pandas as pd
import os
from typing import Optional, List, Dict

_predictor_instance: Optional['Predictor'] = None

class Predictor:
  """Wrapper for model predictions"""

  def __init__(self, meta_path: str):
    self.device = "cuda" if torch.cuda.is_available() else "cpu"
    # Note: Ensure settings or variables for MODEL_PATH and NUM_CLASSES are defined
    # For now, using defaults based on the dataset info
    num_classes = 12144 # Total records in ETL9G_01
    self.model = load_model("model.pth", num_classes=num_classes, device=self.device)
    self.model.to(self.device)

    self.meta_path = meta_path
    self.character_map = self._load_character_map()

  def _load_character_map(self) -> Dict[int, str]:
    """Load character mapping from the ETL9G meta.csv"""
    if os.path.exists(self.meta_path):
      df = pd.read_csv(self.meta_path)
      # Map index (Unnamed: 0 or row index) to the 'char' column
      return df['char'].to_dict()
    else:
      print(f"Warning: Metadata not found at {self.meta_path}")
      return {}

  def predict(self, canvas_data: np.ndarray, top_k: int = 15) -> List[Dict[str, float]]:
    """
    Predict character from canvas data.
    """
    canvas_normalized = canvas_data.astype(np.float32) / 255.0
    input_tensor = torch.from_numpy(canvas_normalized).unsqueeze(0).to(self.device)

    with torch.no_grad():
      logits = self.model(input_tensor)
      probabilities = torch.softmax(logits, dim=1)[0]

    # Using top_k constraint or total classes
    actual_top_k = min(top_k, len(probabilities))
    top_probs, top_indices = torch.topk(probabilities, actual_top_k)

    predictions = []
    for prob, idx in zip(top_probs, top_indices):
      char_idx = idx.item()
      predictions.append({
        "character": self.character_map.get(char_idx, "?"),
        "confidence": float(prob.item())
      })

    return predictions

def get_predictor() -> Predictor:
    """Get or create singleton predictor instance"""
    global _predictor_instance
    if _predictor_instance is None:
        meta_csv = '/content/etlcdb/unpack_etlcdb/ETL9G/ETL9G_01_unpack/meta.csv'
        _predictor_instance = Predictor(meta_csv)
    return _predictor_instance

## 4. Data Loading and Splitting
We define a `JapaneseDataset` class to handle image loading and preprocessing, then split the data into Train, Validation, and Test sets.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import pandas as pd
import os
from PIL import Image
from tqdm.auto import tqdm

class JapaneseDataset(Dataset):
    def __init__(self, base_dir, transform=None):
        self.base_dir = base_dir
        self.transform = transform
        self.all_data = []

        # Find all unpacked directories
        unpack_dirs = sorted([d for d in os.listdir(base_dir) if d.endswith('_unpack')])

        print(f"Loading metadata from {len(unpack_dirs)} volumes...")
        for d in tqdm(unpack_dirs, desc="Volumes"):
            vol_path = os.path.join(base_dir, d)
            meta_file = os.path.join(vol_path, 'meta.csv')
            if os.path.exists(meta_file):
                df = pd.read_csv(meta_file)
                # Store path and label for every image in this volume
                for idx, row in df.iterrows():
                    img_name = f"{idx:05d}.png"
                    img_path = os.path.join(vol_path, img_name)
                    # Class label corresponds to the 'Unnamed: 0' (0-12143)
                    label = int(row['Unnamed: 0'])
                    self.all_data.append((img_path, label))

    def __len__(self):
        return len(self.all_data)

    def __getitem__(self, idx):
        img_path, label = self.all_data[idx]
        image = Image.open(img_path).convert('L')

        if self.transform:
            image = self.transform(image)

        return image, label

# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Initialize Dataset with ALL volumes
base_dir = '/content/etlcdb/unpack_etlcdb/ETL9G'
full_dataset = JapaneseDataset(base_dir, transform=transform)

# Split into Train (80%), Val (10%), Test (10%)
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

# Create DataLoaders
BATCH_SIZE = 128 # Increased batch size for more data
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Total images: {len(full_dataset)}")
print(f"Split: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

In [ ]:
import torch

# 1. Check if CUDA is available at all
cuda_available = torch.cuda.is_available()
print(f"Is CUDA available? {cuda_available}")

if cuda_available:
    # 2. Get the name of the current GPU
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

    # 3. Move the model to GPU
    model.to('cuda')

    # 4. Verify the model's location
    # We check the device of the first parameter
    model_device = next(model.parameters()).device
    print(f"Model is currently on: {model_device}")
else:
    print("CUDA is not available. The model will run on the CPU.")

## 5. Training Loop
We will now define the training and evaluation logic to train our model on the ETL9G dataset.

In [ ]:
import torch.optim as optim
import time

# Re-initialize the model with CNN architecture
num_classes = 12144
model = JapaneseCharacterClassifier(num_classes=num_classes)

# Start training with the new CNN architecture
# We'll stick to 5 epochs for now to check if accuracy starts to move above zero
history = train_model(model, train_loader, val_loader, epochs=5, learning_rate=0.001)